# Channel maps with front-end chips

The veto tiles are read by front-end chips with a limited number of inputs, and the chip outputs are grouped into the boards that form the veto
channels. This notebook shows the model (`analytic/chipLayout.py`) and the search, merging up to a point and then simulated annealing, on the HERD
showcase at $(N_G, N_C) = (5, 8)$:

1. the model and the chip budget;
2. how much each annealing move changes the cost, and why the temperature has to follow the finest move;
3. the cost along the annealing, from different starts;
4. the final costs;
5. the best layout within the chip budget, on the unfolded PSD and in 3D.

The annealing runs are saved in `examples/results/chipLayout.pkl` and reused when the inputs are unchanged (about 3 minutes otherwise).

In [ ]:
import sys, pickle, pathlib, time
sys.path.insert(0, "../analytic")

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, Normalize
from matplotlib.collections import LineCollection
from matplotlib.patches import Patch

import patternDump as pd
import mergeUtils as mu
import rateUtils as ru
import plotUtils as pu
import chipLayout as cl

pu.setStyle()
patterns = pd.selectPatterns(pd.parsePatternDump(pd.defaultDumpPath), **pd.triggerSelections["thesis"])
incidence = pd.buildIncidence(patterns)
M, tileKeys = incidence.matrix, incidence.tileKeys
cost = ru.costMatrix(M.shape[0], ru.tileVetoProbability(tileKeys, window=100e-9))     # 100 ns: Placeholder window (rateUtils.defaultWindow).
referenceCost = float((cost * M).sum())
relativeCost = lambda layout: mu.computeMaskCost(M, layout.channelMap(M), cost)[0] / referenceCost
numberOfGroups, numberOfBoards, capacity = 5, 8, cl.defaultCapacity
neighbours = cl.tileNeighbours(tileKeys)
boardGraph = cl.boardNeighbours(tileKeys, neighbours)                 # Chip graph plus the contacts across cube edges.
minimumChips = cl.minimumChips(tileKeys, capacity)
print(f"{M.shape[0]} patterns x {M.shape[1]} tiles; capacity {capacity} tiles per chip; minimum chips per face {minimumChips}, "
      f"{sum(minimumChips.values())} in total")

## 1. The model

The readout has three levels. A chip ORs at most $M_{\rm chip}$ tiles (32 here, a placeholder that can be set), all on one PSD face and connected in
the neighbour graph of the tiles: two tiles are neighbours when their cells share an edge in the same layer, or when they sit in the same or
edge-sharing cells of the two layers. A board, one of the $N_C$ veto channels, ORs whole chips, and the routing to the $N_G$ pattern groups is the
union routing of the rest of the project. A chip never crosses a face; a board may, where the faces touch on the cube, but it must be one
connected region there (the chip neighbour graph plus the contacts across cube edges). The boards are therefore unions of chips, and the number of chips is a budget: without it the best layout
would be one tile per chip. The budget here is the minimum, the tiles of each face divided by the capacity and rounded up.

The search merges up to a point and then anneals. Simulated annealing proposes a small random change and keeps it if it lowers the cost; if it
raises the cost by $\Delta$ it keeps it with probability $e^{-\Delta/T}$, where $T$ is the temperature, lowered geometrically during the run so
that the search can leave local minima early and settles at the end. The five moves are a tile to a neighbouring chip (the chip it joins stays
within capacity, the one it leaves stays connected), a chip to another board it touches (the board it leaves stays connected), a pattern to
another group, two neighbouring tiles of different chips swapping chips (the only way to change a full chip), and two chips swapping boards.


### The neighbour graph at a corner

The constraints act on a graph of the tiles: a chip must be connected in the graph of neighbours within one face (cells sharing an edge in a
layer, and the same or edge-sharing cells of the other layer), and a board must be connected once the contacts across cube edges are added. The
corner where TOP, Xpos and Ypos meet, about four cells of each face in both layers, shows every kind of link. The top face is 14 tiles wide and the
sides 13, so a top edge tile touches one or two side tiles (the zigzag); neighbouring sides touch row by row.


In [ ]:

from mpl_toolkits.mplot3d.art3d import Line3DCollection
import geometryUtils as gu

def inCorner(face, tileId):
    # About 4 x 4 cells of each face around the corner of TOP, Xpos and Ypos.
    row, column = gu.gridIndex(face, tileId)
    if face == "TOP":
        return row >= 10 and column >= 10
    if face == "Xpos":                                                      # h grows towards Ypos.
        return row >= 9 and column <= 3
    if face == "Ypos":                                                      # h = 0 borders Xpos.
        return row <= 3 and column <= 3
    return False

corner = [index for index, (face, layer, tileId) in enumerate(tileKeys) if inCorner(face, tileId)]
cornerSet = set(corner)
centre = {index: gu.tileFrame(*tileKeys[index])[0] for index in corner}
linkKinds = {"neighbours in a layer": [], "between the two layers": [], "across a TOP-side edge": [], "across a side-side edge": []}
for a in corner:
    for b in boardGraph[1][boardGraph[0][a]:boardGraph[0][a + 1]]:
        if b <= a or b not in cornerSet:
            continue
        (faceA, layerA, _), (faceB, layerB, _) = tileKeys[a], tileKeys[b]
        if faceA == faceB:
            kind = "neighbours in a layer" if layerA == layerB else "between the two layers"
        else:
            kind = "across a TOP-side edge" if "TOP" in (faceA, faceB) else "across a side-side edge"
        linkKinds[kind].append([centre[a], centre[b]])
linkStyle = {"neighbours in a layer": (pu.inkMuted, 0.9, "-"), "between the two layers": (pu.categoricalColours[0], 0.6, ":"),
             "across a TOP-side edge": (pu.categoricalColours[1], 1.6, "-"), "across a side-side edge": (pu.categoricalColours[2], 1.6, "-")}
figure = plt.figure(figsize=(9, 6.5), constrained_layout=True)
drawingAxis = figure.add_subplot(projection="3d")
for kind, segments in linkKinds.items():
    colour, width, line = linkStyle[kind]
    drawingAxis.add_collection3d(Line3DCollection(segments, colors=colour, linewidths=width, linestyles=line, label=f"{kind} ({len(segments)})"))
points = np.array([centre[index] for index in corner])
layers = np.array([tileKeys[index][1] for index in corner])
for layer, marker in ((0, "o"), (1, "s")):
    drawingAxis.scatter(*points[layers == layer].T, s=14, color=pu.inkPrimary if layer == 0 else pu.inkMuted, marker=marker, depthshade=False,
                        label=f"tile centres, layer {layer}")
drawingAxis.set(xlim=(25, 72), ylim=(25, 72), zlim=(-42, 2))
drawingAxis.set_box_aspect((47, 47, 44))
drawingAxis.view_init(elev=30, azim=45)
drawingAxis.set_axis_off()
drawingAxis.legend(loc="upper left", fontsize=9)
drawingAxis.set_title("Neighbour graph at the corner of TOP, Xpos and Ypos", color=pu.inkPrimary)
plt.show()


## 2. How much each move changes the cost

The cost changes $\Delta$ of 20000 proposals of each move type at a start map (serpentine chips, groups from the exact-cost greedy, contiguous
boards merged by exact cost), none accepted. Costs are relative to the cost-weighted size of the RoIs, $\sum_{it} C_{it} M_{it}$.

In [ ]:
cachePath = pathlib.Path("results/chipLayout.pkl")
settings = dict(groups=numberOfGroups, boards=numberOfBoards, capacity=capacity, steps=20_000_000, plateauSteps=300_000, seeds=(0, 1, 2), borderPenalty=1e-4, contiguousBoards=True, swaps=True, version=4)
cached = pickle.loads(cachePath.read_bytes()) if cachePath.exists() else None
useCache = cached is not None and cached["settings"] == settings

rowGroup = cached["rowGroup"] if useCache else mu.mergeGreedily(M, numberOfGroups, M.shape[1], cost, order="groupsFirst").rowGroup
starts = {}
serpentine45 = cl.serpentineChips(tileKeys, minimumChips)
merged51, reached = cl.mergeTilesIntoChips(M, cost, tileKeys, rowGroup, minimumChips, capacity, neighbours)
repaired45, repairedReached = cl.repairChips(M, cost, tileKeys, rowGroup, merged51, minimumChips, capacity, neighbours)
metis45 = cl.metisChips(M, cost, tileKeys, minimumChips, capacity, neighbours)
for name, chipOf in (("tile merge, repaired to 45 chips", repaired45), ("METIS partition, 45 chips", metis45), ("serpentine, 45 chips", serpentine45),
                     ("tile merge, 51 chips (over budget)", merged51)):
    starts[name] = cl.ChipLayout(rowGroup, chipOf, cl.mergeChipsIntoContiguousBoards(M, cost, tileKeys, rowGroup, chipOf, numberOfBoards, boardGraph))
print("tile merge reached", reached, "chips per face against", minimumChips, "; after the repair", repairedReached)

roi, zeroCost = M.astype(np.int64), cost * ~M
start = starts["serpentine, 45 chips"]
chipSize = np.bincount(start.chipOf).astype(np.int64)
moveNames = ("tile to a neighbouring chip", "chip to another board", "pattern to another group", "two tiles swap chips", "two chips swap boards")
proposalDeltas = {}
for moveType, name in enumerate(moveNames):
    shares = np.zeros(len(moveNames)); shares[moveType] = 1.0
    deltas = np.full(20000, np.nan)
    cl._anneal(roi, zeroCost, start.rowGroup.copy(), start.chipOf.copy(), start.boardOfChip.copy(), chipSize.copy(), np.full(numberOfGroups, -1),
               np.zeros(M.shape[0], dtype=np.int64), np.full(numberOfBoards, -1), np.zeros(chipSize.size, dtype=np.int64), neighbours[0], neighbours[1],
               numberOfGroups, numberOfBoards, capacity, 60000, 1.0, 1.0, shares, 7 + moveType, 60000, deltas, chipSize.size, 0.0, 0.0, True,
               boardGraph[0], boardGraph[1])
    proposalDeltas[name] = deltas[np.isfinite(deltas)] / referenceCost

moveColours = dict(zip(moveNames, pu.categoricalColours[:5]))
figure, drawingAxis = plt.subplots(figsize=(9, 4.6), constrained_layout=True)
bins = np.logspace(-6, 0.5, 60)
for name, deltas in proposalDeltas.items():
    uphill = deltas[deltas > 1e-12]
    drawingAxis.hist(uphill, bins=bins, histtype="step", linewidth=2, color=moveColours[name],
                     label=f"{name}: {np.mean(deltas < -1e-12):.1%} downhill, {np.mean(np.abs(deltas) <= 1e-12):.0%} neutral")
drawingAxis.set(xscale="log", xlabel=r"uphill cost change $\Delta$ / $\sum C M$", ylabel="proposals", title="Cost change of single moves at the start map")
drawingAxis.legend(fontsize=9, loc="upper left")
plt.show()
for name, deltas in proposalDeltas.items():
    print(f"{name}: median uphill change {np.median(deltas[deltas > 1e-12]):.2g}")

The moves differ in scale by three orders of magnitude. A temperature that lets a median chip or pattern move through also lets almost any tile
move through, and under union routing the map then drifts within a few thousand steps onto the plateau where every group is vetoed by every board:
there most moves cost nothing, and no single move leads back. The start temperature is therefore set from the tile moves (a median uphill tile move
kept with probability 0.1); uphill chip and pattern moves are then almost never kept, but their downhill ones are.

## 3. The cost along the annealing

Each start is annealed from three random seeds for $2\times10^7$ steps, with the chip border penalty of section 5. The starts are the tile merge repaired to the budget (the smallest chips
dissolved into their neighbours, through chains of neighbouring chips when those are full), a METIS partition of each face weighted by the
similarity of the tiles' pattern sets, a serpentine tiling that ignores the cost, and the tile merge without repair, which stays above the budget.
One more run, from the serpentine start, sets the temperature from all moves, to show the plateau.

In [ ]:
runs = {} if not useCache else cached["runs"]
penaltyScan = None if not useCache else cached["penaltyScan"]
swapComparison = None if not useCache else cached["swapComparison"]
startColours = {"tile merge, repaired to 45 chips": pu.categoricalColours[2], "METIS partition, 45 chips": pu.categoricalColours[4],
                "serpentine, 45 chips": pu.categoricalColours[0], "tile merge, 51 chips (over budget)": pu.categoricalColours[1],
                "temperature from all moves": pu.categoricalColours[6]}
if not useCache:
    began = time.time()
    plans = [(name, starts[name], settings["steps"], 0.1, 45 if "45" in name else None) for name in starts]
    for name, startLayout, steps, acceptance, budget in plans:
        for seed in settings["seeds"]:
            layout, _, info = cl.annealLayout(M, cost, tileKeys, startLayout, numberOfGroups, numberOfBoards, capacity, steps, seed,
                                              startAcceptance=acceptance, chipBudget=budget, borderPenalty=settings["borderPenalty"] * referenceCost,
                                              contiguousBoards=True, neighbours=neighbours, boardGraph=boardGraph)
            cl.checkLayout(layout, tileKeys, capacity, neighbours, contiguousBoards=True)
            runs[(name, seed)] = dict(layout=layout, cost=relativeCost(layout), startCost=relativeCost(startLayout), chips=layout.numberOfChips,
                                      chipBorders=cl.chipBorderLength(layout, tileKeys, neighbours), trace=info["trace"],
                                      temperatures=info["temperatures"])
    plateauTemperature = float(np.median(np.concatenate([d[d > 1e-12] for d in proposalDeltas.values()])) * referenceCost / np.log(10))
    layout, _, info = cl.annealLayout(M, cost, tileKeys, starts["serpentine, 45 chips"], numberOfGroups, numberOfBoards, capacity,
                                      settings["plateauSteps"], 0, temperatures=(plateauTemperature, plateauTemperature * 1e-3),
                                      contiguousBoards=True, neighbours=neighbours, boardGraph=boardGraph)
    runs[("temperature from all moves", 0)] = dict(layout=layout, cost=relativeCost(layout), startCost=relativeCost(starts["serpentine, 45 chips"]),
                                                   chips=layout.numberOfChips, trace=info["trace"], temperatures=info["temperatures"])
    cachePath.parent.mkdir(exist_ok=True)
    swapComparison = {}
    for label, shares in (("single moves only", (0.5, 0.2, 0.3, 0.0, 0.0)), ("with swaps", (0.35, 0.15, 0.2, 0.2, 0.1))):
        for seed in settings["seeds"]:
            layout, _, _ = cl.annealLayout(M, cost, tileKeys, starts["tile merge, repaired to 45 chips"], numberOfGroups, numberOfBoards, capacity,
                                           settings["steps"], seed, moveShares=shares, chipBudget=45, borderPenalty=settings["borderPenalty"] * referenceCost,
                                           contiguousBoards=True, neighbours=neighbours, boardGraph=boardGraph)
            swapComparison[(label, seed)] = relativeCost(layout)
    penaltyScan = {}
    for penalty in (0.0, 1e-5, 3e-5, 1e-4, 3e-4):                        # Chip border penalty, times sum C M per border.
        for seed in (0, 1):
            layout, _, _ = cl.annealLayout(M, cost, tileKeys, starts["tile merge, repaired to 45 chips"], numberOfGroups, numberOfBoards, capacity,
                                           settings["steps"], seed, chipBudget=45, borderPenalty=penalty * referenceCost, contiguousBoards=True,
                                           neighbours=neighbours, boardGraph=boardGraph)
            penaltyScan[(penalty, seed)] = (relativeCost(layout), cl.chipBorderLength(layout, tileKeys, neighbours))
    cachePath.write_bytes(pickle.dumps(dict(settings=settings, rowGroup=rowGroup, runs=runs, penaltyScan=penaltyScan, swapComparison=swapComparison)))
    print(f"annealing runs: {time.time() - began:.0f} s")

figure, drawingAxis = plt.subplots(figsize=(11, 5.6), constrained_layout=True)
labelled = set()
for (name, seed), run in runs.items():
    trace = run["trace"][run["trace"][:, 0] > 0]
    drawingAxis.plot(trace[:, 0], trace[:, 2] / referenceCost, color=startColours[name], linewidth=1.4, alpha=0.9,
                     label=name if name not in labelled else None)
    labelled.add(name)
drawingAxis.axhline(2.05, color=pu.inkPrimary, linestyle="--", linewidth=1)
drawingAxis.annotate("best (5, 8) map without chips", (2e2, 2.05), xytext=(0, -14), textcoords="offset points", fontsize=9, color=pu.inkPrimary)
drawingAxis.set(xscale="log", xlabel="annealing step", ylabel=r"current cost / $\sum C M$", title="Cost along the annealing, (5, 8), three seeds per start")
drawingAxis.legend(fontsize=9, loc="center left")
plt.show()

## 4. Final costs

The best cost of each run (dots), its start (crosses), and the number of chips it ends with. The unrepaired tile merge ends above the budget of
45: the annealing penalty per extra chip cannot drain chips whose neighbours are full, which is what the repair does.

Below the plot, the repaired start annealed with the three single moves only and with the two swap moves added, three seeds each.

In [ ]:
names = [name for name in startColours if name != "temperature from all moves"]
figure, drawingAxis = plt.subplots(figsize=(10, 4.6), constrained_layout=True)
for row, name in enumerate(names):
    entries = [run for (runName, _), run in runs.items() if runName == name]
    drawingAxis.plot([entry["startCost"] for entry in entries], [row] * len(entries), "x", color=startColours[name], markersize=9)
    drawingAxis.plot([entry["cost"] for entry in entries], [row] * len(entries), "o", color=startColours[name], markersize=9)
    drawingAxis.annotate(", ".join(str(entry["chips"]) for entry in entries) + " chips", (max(entry["cost"] for entry in entries), row),
                         xytext=(8, 6), textcoords="offset points", fontsize=9, color=pu.inkPrimary)
drawingAxis.axvline(2.05, color=pu.inkPrimary, linestyle="--", linewidth=1)
drawingAxis.set(yticks=range(len(names)), yticklabels=names, xlabel=r"cost / $\sum C M$ (x: start, o: after annealing)",
                title="Final cost per start and seed, (5, 8); dashed: the best map without chips")
drawingAxis.invert_yaxis()
plt.show()
for name in names:
    entries = [run for (runName, _), run in runs.items() if runName == name]
    print(f"{name:42s} start {entries[0]['startCost']:.3f} -> " + ", ".join(f"{e['cost']:.3f} ({e['chips']} chips)" for e in entries))
for label in ("single moves only", "with swaps"):
    values = [swapComparison[(label, seed)] for seed in settings["seeds"]]
    print(f"repaired start, {label:18s}: " + ", ".join(f"{value:.4f}" for value in values) + f"  (mean {np.mean(values):.4f})")



## 5. Chip shapes

The veto cost does not depend on the shape of a chip as long as its tiles stay on the same board, so two thirds of the tile moves change nothing
and, accepted freely, let the chips drift into winding shapes; longer annealing makes them more irregular, not less. A small penalty per chip
border (a pair of neighbouring tiles on different chips) breaks these ties. The scan below anneals the repaired start for $2\times10^7$ steps with
penalties from 0 to $3\times10^{-4}\sum C M$ per border, two seeds each: the veto cost against the number of chip borders. A layout of compact
blocks of 4 x 4 cells in both layers would have about 1000 borders.


In [ ]:

figure, drawingAxis = plt.subplots(figsize=(8.5, 5), constrained_layout=True)
penalties = sorted({penalty for penalty, _ in penaltyScan})
for colour, penalty in zip(pu.ordinalRamp[1:], penalties):
    points = np.array([penaltyScan[(penalty, seed)] for seed in (0, 1)])
    drawingAxis.plot(points[:, 1], points[:, 0], "o", color=colour, markersize=9, label=f"{penalty:.0e}" if penalty else "none")
drawingAxis.set(xlabel="chip borders (pairs of neighbouring tiles on different chips)", ylabel=r"veto cost / $\sum C M$",
                title="Chip border penalty: compactness against veto cost, (5, 8), 45 chips")
drawingAxis.legend(title=r"penalty per border / $\sum C M$", fontsize=9)
plt.show()
for penalty in penalties:
    print(f"penalty {penalty:.0e}: " + ", ".join(f"cost {penaltyScan[(penalty, seed)][0]:.4f}, {penaltyScan[(penalty, seed)][1]} borders" for seed in (0, 1)))


## 6. The best layout within the budget

The lowest-cost run that ends within 45 chips, after a deterministic polish (sweeps over the chip perimeters testing every tile move, tile swap,
double swap, ejection chain (a tile joins a full chip, which passes a tile along neighbouring chips to one with room), chip move and swap and
pattern move until none lowers the cost), on the unfolded PSD (both layers): each tile coloured by its board, in a shade of its own per chip, and a line wherever two
neighbouring tiles of a layer belong to different chips; every face edge is
drawn too, since no chip crosses a face.

In [ ]:
import geometryUtils as gu
withinBudget = {key: run for key, run in runs.items() if run["chips"] <= sum(minimumChips.values()) and key[0] != "temperature from all moves"}
bestKey = min(withinBudget, key=lambda key: withinBudget[key]["cost"])
best, polishChanges, polishHistory = cl.polishLayout(M, cost, tileKeys, withinBudget[bestKey]["layout"], numberOfGroups, numberOfBoards, capacity,
                                                     settings["borderPenalty"] * referenceCost, True, sum(minimumChips.values()),
                                                     neighbours=neighbours, boardGraph=boardGraph)
cl.checkLayout(best, tileKeys, capacity, neighbours, contiguousBoards=True)
print(f"polish: {sum(polishChanges.values())} changes {polishChanges}, cost {polishHistory[0]:.4f} -> {polishHistory[-1]:.4f} in {len(polishHistory) - 1} sweeps")
print(f"best within {sum(minimumChips.values())} chips: {bestKey[0]}, seed {bestKey[1]}, cost {relativeCost(best):.4f} after the polish, "
      f"{best.numberOfChips} chips, sizes {np.bincount(best.chipOf).min()}-{np.bincount(best.chipOf).max()}")

def chipBorders(layout, layer):
    # Segments on the unfolded cross between same-layer neighbouring tiles of different chips, and the face edges that touch another face.
    segments = []
    keyIndex = {key: index for index, key in enumerate(tileKeys)}
    for index, (face, tileLayer, tileId) in enumerate(tileKeys):
        if tileLayer != layer:
            continue
        for other in neighbours[1][neighbours[0][index]:neighbours[0][index + 1]]:
            otherFace, otherLayer, otherId = tileKeys[other]
            if otherLayer == layer and other > index and layout.chipOf[other] != layout.chipOf[index]:
                first, second = pu.unfoldedPosition(face, tileId), pu.unfoldedPosition(otherFace, otherId)
                middle, along = (first + second) / 2, (second - first)
                across = np.array([-along[1], along[0]]) / 2
                segments.append([middle - across, middle + across])
    segments += faceContactEdges()
    return segments

def faceContactEdges():
    # Every face edge: TOP meets the four sides, each side meets its two neighbours and, at its bottom, the edge of the detector. No chip
    # crosses a face, so all of them are chip borders.
    edges = []
    for face in pd.psdSides:
        centres = np.array([pu.unfoldedPosition(face, tileId) for tileId in range(pd.tilesPerLayer[face])])
        (left, bottom), (right, top) = centres.min(axis=0) - 0.5, centres.max(axis=0) + 0.5
        edges += [[(left, bottom), (right, bottom)], [(right, bottom), (right, top)], [(right, top), (left, top)], [(left, top), (left, bottom)]]
    return edges

def chipShades(layout):
    # A colour per chip: its board's colour, lightened or darkened by a step that differs between the chips of the board.
    from matplotlib.colors import to_rgb
    shades = {}
    for board in np.unique(layout.boardOfChip):
        chips = np.flatnonzero(layout.boardOfChip == board)
        base = np.array(to_rgb(pu.categoricalColours[board]))
        for position, chip in enumerate(chips):
            step = np.linspace(-0.35, 0.45, len(chips))[position] if len(chips) > 1 else 0.0
            shades[chip] = base + (1 - base) * step if step > 0 else base * (1 + step)
    return shades

boardColours = ListedColormap(pu.categoricalColours[:numberOfBoards])
tileBoard = best.boardOfChip[best.chipOf]
figure, axes = plt.subplots(1, 2, figsize=(15, 7), constrained_layout=True)
shades = chipShades(best)
chipColours = ListedColormap([shades[chip] for chip in range(best.chipOf.max() + 1)])
for layer in (0, 1):
    valueOf = {key: best.chipOf[index] for index, key in enumerate(tileKeys)}
    pu.plotUnfoldedPsd(axes[layer], valueOf, layer, Normalize(-0.5, best.chipOf.max() + 0.5), chipColours, f"PSD layer {layer}")
    axes[layer].add_collection(LineCollection(chipBorders(best, layer), colors=pu.inkPrimary, linewidths=1.8))
figure.legend(handles=[Patch(facecolor=pu.categoricalColours[board], label=f"board {board}: {np.isin(best.chipOf, np.flatnonzero(best.boardOfChip == board)).sum()} tiles, "
                             f"{np.unique(best.chipOf[tileBoard == board]).size} chips") for board in range(numberOfBoards)],
              loc="outside right center", title="boards (veto channels)")
figure.suptitle(f"Best layout within {sum(minimumChips.values())} chips of {capacity} inputs, (N_G, N_C) = (5, 8): colour = board, shade = chip, lines = chip borders",
                color=pu.inkPrimary)
plt.show()

figure = plt.figure(figsize=(15, 6), constrained_layout=True)
colourOf = {(face, tileId): shades[best.chipOf[index]] for index, (face, layer, tileId) in enumerate(tileKeys) if layer == 0}
for column, azimuth in enumerate((-60, 120)):
    pu.plotPsdCube(figure.add_subplot(1, 2, column + 1, projection="3d"), colourOf, azimuth=azimuth, title=f"layer 0, seen from azimuth {azimuth}°")
plt.show()
pieces = {board: cl._isConnected(np.flatnonzero(tileBoard == board), *boardGraph) for board in range(numberOfBoards)}
print("every board connected on the cube:", all(pieces.values()))

## 7. Where this stands

- Annealing works once its temperature follows the tile moves; set from all moves, it sends the map onto the plateau at 7.33 within a few hundred
  steps and never returns.
- The start decides the result. With boards contiguous on the cube, 45 chips of 32 inputs and the chip border penalty, the tile merge repaired to
  the budget ends at 2.178 to 2.183, the METIS partition at 2.535 to 2.554 and the serpentine tiling at 3.02 to 3.05, three seeds each. Free
  boards reach 2.11 and the best $(5, 8)$ map without chips 2.05: the minimum chip budget and contiguous boards together cost about 6%.
- Swap moves lower the result by about 0.3%. The deterministic polish adds about 0.1% (2.178 to 2.176), mostly through ejection chains: the
  annealing ends near a local optimum for tile moves, swaps, double swaps, chip moves and pattern moves. Of the remaining perimeter moves most are blocked by full chips (25 of 45 hold
  32 tiles) or raise the veto cost; ignoring connectivity, only two swaps would lower the cost, by 0.02% together.
- One more chip on Xpos (46 in total) ends at 2.124 and straightens the board boundaries there: the chip budget is the main remaining lever,
  a trade-off between chips and veto cost that a cost per chip in the objective would make part of the optimisation.
- Open: the cost against the chip budget, the other $(N_G, N_C)$ budgets, and the L1 efficiency of the layouts on the photons.